<!--
Artifact: ML_RPM_demo.ipynb
Author: Obed Plasencia
Date coded: November 12, 2026
-->
# ML-RPM demo: a walkthrough of each use case

This notebook runs the ML-RPM package one use case at a time, with the demo wells in `data/`.
It does the same work as the command-line tool (`python -m mlrpm ...`), step by step, so you can
look at the result of each step.

**Before you start:** open this notebook from the repository folder or from `examples/`, and install
the requirements (`pip install -r requirements.txt`).

> The demo wells are synthetic stand-ins for field data (see `examples/make_demo_data.py`).
> The scores show that the system works. They are not findings about real rocks.

In [1]:
# Find the repository root, so the notebook works from the repo folder or from examples/.
import os, sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "mlrpm").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import matplotlib
matplotlib.use("Agg")            # draw figures without opening windows
import matplotlib.pyplot as plt
from IPython.display import display

from mlrpm import (RockPhysicsPipeline, WellLogRepository, ForwardModel,
                   SyntheticWellGenerator, load_config)
from mlrpm.plotting import LogPlotter, ResultsVisualizer

cfg = load_config()              # every setting comes from default_config.yaml (NFR-10)
DATA = ROOT / "data"
print("Repository folder:", ROOT.name)

Repository folder: rock-physics-model-sweng3321


## UC-1: Upload well data

`WellLogRepository` reads each `<well>.csv` file, maps its column headers to the internal names
with `column_map` in the config (FR-1, NFR-6), and checks that every required column is there and
is numeric (FR-2).

In [2]:
repo = WellLogRepository(DATA, cfg["column_map"])
names = repo.available_wells()
print("Wells found:", names)

wells = repo.load_many(names)
for name, w in wells.items():
    print(f"{name}: {len(w)} samples, depth {w.depth().min():.0f}-{w.depth().max():.0f} m")

wells[names[0]].to_dataframe().head()

Wells found: ['WELL-A', 'WELL-B', 'WELL-C', 'WELL-D']
WELL-A: 600 samples, depth 2500-2800 m
WELL-B: 600 samples, depth 2650-2950 m
WELL-C: 600 samples, depth 2800-3100 m
WELL-D: 600 samples, depth 2950-3250 m


,DEPTH,VSH,VSAND,VCARB,POR,SW,PD,VP,VS,RHO
0,2500.000000,0.348051,0.523214,0.128734,0.242971,0.432460,15.623039,2.121687,1.315817,2.150468
1,2500.500835,0.359602,0.512975,0.127423,0.241198,0.431915,15.556111,2.108681,1.269145,2.132707
2,2501.001669,0.351767,0.521030,0.127204,0.245810,0.439864,15.477001,2.096503,1.261943,2.111558
3,2501.502504,0.357731,0.510555,0.131713,0.243929,0.431776,15.168217,2.049985,1.258390,2.116396
4,2502.003339,0.354421,0.518908,0.126671,0.241851,0.461935,14.686523,2.053353,1.285902,2.118539


## UC-2: Run the theoretical forward model (the RPM)

`ForwardModel` predicts Vp and Vs from the six inputs with Hertz-Mindlin (dry frame),
Batzle-Wang (fluids) and Gassmann (fluid substitution) (FR-4). Here it runs on one well and the
prediction is plotted against the measured log.

In [3]:
fm = ForwardModel.from_config(cfg)
w = wells[names[0]]
vp_rpm = fm.predict(w, "VP")

plotter = LogPlotter()
fig, ax = plt.subplots(figsize=(4, 7))
plotter.plot_depth_track(w.depth(), w.target("VP"), {"RPM": vp_rpm}, "VP", w.name, ax=ax)
display(fig); plt.close(fig)

<Figure size 400x700 with 1 Axes>

## UC-5: Generate synthetic wells

`SyntheticWellGenerator` builds extra wells from the forward model (FR-5). It uses a fixed random
seed, so the same settings always give the same wells (NFR-11). Synthetic wells are only used as
training data, never as the blind well (NFR-12).

In [4]:
gen = SyntheticWellGenerator.from_config(fm, cfg)
synthetic = gen.generate(3)
for name, s in synthetic.items():
    print(f"{name}: {len(s)} samples, mean Vp {s.target('VP').mean():.2f} km/s")

SYN-001: 500 samples, mean Vp 2.26 km/s
SYN-002: 500 samples, mean Vp 2.28 km/s
SYN-003: 500 samples, mean Vp 2.30 km/s


## UC-9: Run the blind-well pipeline

`RockPhysicsPipeline.run` does the whole evaluation in one call (FR-12). Each real well takes a
turn as the blind well: the ML models (SVR, RF, MLP) train on the other wells (plus the synthetic
wells), and then every method, including the RPM, predicts the blind well (FR-6, FR-7).

In [5]:
pipe = RockPhysicsPipeline.from_config(DATA, cfg)
res_vp = pipe.run(target_name="VP", n_synthetic=3)
res_vs = pipe.run(target_name="VS")            # VS was run without synthetic wells in the report
print("Done. Blind wells:", list(res_vp.blind_wells))

Done. Blind wells: ['WELL-A', 'WELL-B', 'WELL-C', 'WELL-D']


## UC-7: Compare the models

`ModelEvaluator` scores every method on every blind well with R², accuracy %, RMSE and MAE
(FR-8). The tables show R² for each blind well, then the averages for each model.

In [6]:
print("VP: R² by blind well")
display(res_vp.report.pivot("r2").round(3))
print("VP: averages by model")
display(res_vp.report.to_dataframe().groupby("model_name")[["r2", "accuracy_pct", "rmse", "mae"]].mean().round(3))

VP: R² by blind well


model_name,RPM,SVR,RF,MLP
well_name,,,,
WELL-A,0.912,0.877,0.830,0.850
WELL-B,0.860,0.907,0.883,0.927
WELL-C,0.714,0.909,0.895,0.962
WELL-D,0.609,0.831,0.892,0.908


VP: averages by model


,r2,accuracy_pct,rmse,mae
model_name,,,,
MLP,0.912,95.797,0.158,0.121
RF,0.875,94.980,0.192,0.152
RPM,0.774,92.940,0.242,0.219
SVR,0.881,95.264,0.184,0.138


In [7]:
print("VS: averages by model")
display(res_vs.report.to_dataframe().groupby("model_name")[["r2", "accuracy_pct", "rmse", "mae"]].mean().round(3))

VS: averages by model


,r2,accuracy_pct,rmse,mae
model_name,,,,
MLP,0.907,95.461,0.096,0.074
RF,0.867,94.701,0.115,0.090
RPM,0.660,91.144,0.172,0.147
SVR,0.897,95.647,0.097,0.073


In [8]:
viz = ResultsVisualizer()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
viz.metric_bar_chart(res_vp.report.pivot("r2"), "R² (VP)", ax=axes[0])
viz.box_plot(res_vp.errors_by_model(), "VP error (km/s)", ax=axes[1])
fig.tight_layout(); display(fig); plt.close(fig)

<Figure size 1200x400 with 2 Axes>

In [9]:
fig = plotter.plot_pipeline_result(res_vp)   # depth tracks for every blind well (FR-9)
display(fig); plt.close(fig)

<Figure size 1200x800 with 4 Axes>

## UC-8: Configure the display (colorblind setting)

The same charts can use a colorblind-friendly palette (UIR-4). On the command line this is the
`--colorblind` option.

In [10]:
viz_cb = ResultsVisualizer(colorblind=True)
fig, ax = plt.subplots(figsize=(6, 4))
viz_cb.metric_bar_chart(res_vs.report.pivot("r2"), "R² (VS), colorblind palette", ax=ax)
fig.tight_layout(); display(fig); plt.close(fig)

<Figure size 600x400 with 1 Axes>

## UC-3: Export the results

The results are saved as CSV tables, the same files that `python -m mlrpm run --out results`
writes (FR-10). The PDF summary was deferred.

In [11]:
out = ROOT / "results_notebook"
out.mkdir(exist_ok=True)
for res in (res_vp, res_vs):
    t = res.target_name
    res.report.to_dataframe().to_csv(out / f"{t}_all_results.csv", index=False)
    for metric in ("r2", "accuracy_pct", "rmse", "mae"):
        res.report.pivot(metric).to_csv(out / f"{t}_{metric}_by_well.csv")
print(sorted(p.name for p in out.glob("*.csv")))

['VP_accuracy_pct_by_well.csv', 'VP_all_results.csv', 'VP_mae_by_well.csv', 'VP_r2_by_well.csv', 'VP_rmse_by_well.csv', 'VS_accuracy_pct_by_well.csv', 'VS_all_results.csv', 'VS_mae_by_well.csv', 'VS_r2_by_well.csv', 'VS_rmse_by_well.csv']
